In [1]:
import psycopg2

conn = psycopg2.connect(
    dbname="bird",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)
cur = conn.cursor()
cur.execute("SELECT version();")
print("Versi PostgreSQL:", cur.fetchone())

cur.close()
conn.close()

Versi PostgreSQL: ('PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit',)


In [3]:
import json

with open("../data/MINIDEV/mini_dev_postgresql.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Tampilkan 3 contoh Gold SQL pertama
for i, item in enumerate(data[:3]):
    print(f"--- Example {i+1} ---")
    print("Question:", item["question"])
    print("DB ID   :", item["db_id"])
    print("Gold SQL:", item["SQL"])
    print("Evidence:", item.get("evidence", ""))

--- Example 1 ---
Question: What is the ratio of customers who pay in EUR against customers who pay in CZK?
DB ID   : debit_card_specializing
Gold SQL: SELECT CAST(SUM(CASE WHEN Currency = 'EUR' THEN 1 ELSE 0 END) AS REAL) / NULLIF(SUM(CASE WHEN Currency = 'CZK' THEN 1 ELSE 0 END), 0) FROM customers
Evidence: ratio of customers who pay in EUR against customers who pay in CZK = count(Currency = 'EUR') / count(Currency = 'CZK').
--- Example 2 ---
Question: In 2012, who had the least consumption in LAM?
DB ID   : debit_card_specializing
Gold SQL: SELECT T1.CustomerID FROM customers AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T1.Segment = 'LAM' AND SUBSTR(T2.Date, 1, 4) = '2012' GROUP BY T1.CustomerID ORDER BY SUM(T2.Consumption) ASC NULLS FIRST LIMIT 1
Evidence: Year 2012 can be presented as Between 201201 And 201212; The first 4 strings of the Date values in the yearmonth table can represent year.
--- Example 3 ---
Question: What was the average monthly consum

In [4]:
import json
import pandas as pd

with open("../data/MINIDEV/mini_dev_postgresql.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

df = df.set_index("question_id")
df.head(20)

,db_id,question,evidence,SQL,difficulty
question_id,,,,,
1471,debit_card_specializing,What is the ratio of customers who pay in EUR ...,ratio of customers who pay in EUR against cust...,SELECT CAST(SUM(CASE WHEN Currency = 'EUR' THE...,simple
1472,debit_card_specializing,"In 2012, who had the least consumption in LAM?",Year 2012 can be presented as Between 201201 A...,SELECT T1.CustomerID FROM customers AS T1 INNE...,moderate
1473,debit_card_specializing,What was the average monthly consumption of cu...,Average Monthly consumption = AVG(Consumption)...,"SELECT AVG(T2.Consumption) / NULLIF(12, 0) FRO...",moderate
1476,debit_card_specializing,What was the difference in gas consumption bet...,cast the consumption into float when perform c...,SELECT SUM(CASE WHEN T1.Currency = 'CZK' THEN ...,challenging
1479,debit_card_specializing,Which year recorded the most consumption of ga...,The first 4 strings of the Date values in the ...,"SELECT SUBSTR(T2.Date, 1, 4) FROM customers AS...",moderate
1480,debit_card_specializing,What was the gas consumption peak month for SM...,Year 2013 can be presented as Between 201301 A...,"SELECT SUBSTR(T2.Date, 5, 2) FROM customers AS...",moderate
1481,debit_card_specializing,What is the difference in the annual average c...,annual average consumption of customer with th...,SELECT CAST(SUM(CASE WHEN T1.Segment = 'SME' T...,challenging
1482,debit_card_specializing,"Which of the three segments—SME, LAM and KAM—h...",Increase or Decrease = consumption for 2013 - ...,SELECT CAST((SUM(CASE WHEN T1.Segment = 'SME' ...,challenging
1483,debit_card_specializing,How much did customer 6 consume in total betwe...,Between August And November 2013 refers to Bet...,SELECT SUM(Consumption) FROM yearmonth WHERE C...,simple
